# Taller 1. Formulación y resolución de un modelo en red

**Investigación de Operaciones**
Doctorado en Ingeniería, consorcio UV–UTA
Jueves 10 de septiembre de 2026, Bloque B

---

**Integrantes:**
**Instancia asignada:**
**Fecha:**

---

Esta plantilla entrega la estructura común a las cuatro instancias. Lo que falta —y es lo que se
evalúa— son la formulación, la interpretación y la verificación. Los bloques marcados con
`# TODO` deben completarse.

Regla dura del taller: **ningún dato numérico se escribe dentro del modelo**. Todo se lee de los
archivos CSV de la carpeta `datos/`.


## Paso 0. Verificación del entorno


In [3]:
import sys
print("Python", sys.version.split()[0])

import pyomo.environ as pyo
print("Pyomo", pyo.__version__ if hasattr(pyo, "__version__") else "instalado")

solver = pyo.SolverFactory("appsi_highs")
print("HiGHS disponible:", solver.available())

import pandas as pd
print("pandas", pd.__version__)

import networkx as nx
print("networkx", nx.__version__)

Python 3.14.7
Pyomo instalado
HiGHS disponible: True
pandas 3.0.5
networkx 3.6.1


## Paso 1. Lectura de los datos

Ajuste `CARPETA` a la instancia que le fue asignada. El separador de los CSV es el punto y coma.


In [4]:
from pathlib import Path
import pandas as pd

CARPETA = Path("datos")          # TODO: apunte a la carpeta de su instancia

# Instancias A y B: nodos.csv y arcos.csv
# Instancia C: demanda.csv y parametros.csv  -> hay que construir la red expandida en el tiempo
# Instancia D: tiempos.csv                   -> hay que construir la red bipartita

nodos = pd.read_csv(CARPETA / "nodos.csv", sep=";")
arcos = pd.read_csv(CARPETA / "arcos.csv", sep=";")
display(nodos)
display(arcos)


,nodo,descripcion,flujo_exogeno_t_mes,tipo
0,F1,Faena Norte (planta concentradora),12000,oferta
1,F2,Faena Centro,9000,oferta
2,F3,Faena Sur,7000,oferta
3,A1,Acopio Calama,0,transbordo
4,A2,Acopio Copiapó,0,transbordo
5,P1,Puerto Angamos,-11000,demanda
6,P2,Puerto Chañaral,-8000,demanda
7,P3,Puerto Ventanas,-6000,demanda


,origen,destino,costo_peso_por_t,cap_min_t,cap_max_t
0,F1,A1,4200,0,10000
1,F1,A2,7800,0,6000
2,F2,A1,5100,0,8000
3,F2,A2,4600,0,8000
4,F3,A1,8400,0,4000
5,F3,A2,3900,0,7000
6,A1,P1,2800,0,12000
7,A1,P2,6100,0,6000
8,A1,P3,9200,0,5000
9,A2,P2,3300,0,9000


## Paso 2. Estructura de la red

Antes de programar nada, escriba en palabras qué representa cada nodo y cada arco, y con qué
unidades. Un modelo cuyas unidades no cierran está mal aunque el solver entregue un número.


Nodos ($N$):
+ Representa las instalaciones (faenas, acopios, puertos).
+ El parámetro de flujo exógeno $q_i$ indica la oferta o demanda de cada punto.
+ La unidad de $q_i$ es toneladas por mes (t/mes) [$\frac{T}{M}$].

Arcos ($A$):
+ Representa las rutas de transporte del mineral entre las instalaciones.
+ Tienen asociados un capacidad máxima de transporte (t/mes) y un costo logístico en unidades de pesos por tonelada ($/t)

In [5]:
# TODO: complete los tres diccionarios

# conjunto de nodos
N = dict(zip(nodos['nodo'], nodos['descripcion']))    
# flujo exógeno por nodo: positivo oferta, negativo demanda, cero transbordo
q = dict(zip(nodos['nodo'], nodos['flujo_exogeno_t_mes']))
# arcos: (i, j) -> (costo, cota_inferior, cota_superior)   
A = { (r['origen'], r['destino']): (r['costo_peso_por_t'], r['cap_min_t'], r['cap_max_t']) for _, r in arcos.iterrows() }

# Comprobación imprescindible antes de seguir:
# si todas las restricciones son igualdades, la suma de los q_i debe ser cero.
print("suma de los flujos exógenos:", sum(q.values()))


suma de los flujos exógenos: 3000


Se supone que la sume debería ser cero ($q_i=0$), pero esto no ocurre. \
La suma de los flujos exógenos es $q_i=$ 3.000 t/mes. \
Esto revela un exceso de oferta en la red, donde la capacidad total de las faenas (28000 t/mes) supera el requerimiento de los puertos (25.000 t/mes). \
Debido a este desbalance, la conservación de flujo en los nodos de origen debe modelarse como una desigualdad ($\le$). \
Si mantenemos el modelo original, estaríamos forzando una igualdad ($=$) y esto obligaría a despachar flujo hacia afuera de la red, haciendo que el sistema sea matemáticamente no factible.

## Paso 3. El modelo

$$\min \; \sum_{(i,j)\in A} c_{ij}\,x_{ij}
\quad\text{s.a.}\quad
\sum_{j} x_{ij} - \sum_{k} x_{ki} = q_i \;\; \forall i \in N,
\qquad l_{ij} \le x_{ij} \le u_{ij}$$

Note que las variables se declaran **continuas**. No se impone integralidad: si el modelo es
realmente de red y el lado derecho es entero, la solución saldrá entera por sí sola. Comprobarlo es
parte del taller.


In [6]:
import pyomo.environ as pyo

m = pyo.ConcreteModel()
m.N = pyo.Set(initialize=list(N))
m.A = pyo.Set(initialize=list(A), dimen=2)

m.x = pyo.Var(m.A, domain=pyo.NonNegativeReals,
              bounds=lambda m, i, j: (A[(i, j)][1], A[(i, j)][2]))

m.obj = pyo.Objective(expr=sum(A[a][0] * m.x[a] for a in m.A), sense=pyo.minimize)

def balance(m, n):
    sale  = sum(m.x[i, j] for (i, j) in m.A if i == n)
    entra = sum(m.x[i, j] for (i, j) in m.A if j == n)
    # TODO: decida para qué nodos corresponde igualdad y para cuáles desigualdad,
    #       y justifique esa decisión en el informe.
    if q[n] > 0:
        return sale - entra <= q[n]
    else:
        return sale - entra == q[n]
    
m.bal = pyo.Constraint(m.N, rule=balance)

# Los valores duales deben declararse ANTES de resolver, o no se importan.
m.dual = pyo.Suffix(direction=pyo.Suffix.IMPORT)

print("variables:", len(m.A), " restricciones:", len(m.N))


variables: 13  restricciones: 8


El modelo planteado:
+ Nodos de oferta (faenas): se define <= porque hay un exceso de oferta en la red (28.000 vs 25.000). Esto permite que el excedente (3.000) quede como capacidad ociosa sin volver el modelo infactible.
+ Nodos de transbordo y demanda (acopios y puertos): se define como == para exigir una conservación estricta, es decir, todo lo que entra debe salir o consumirse.
+ 13 variables = 13 arcos en la red (rutas posibles)
+ 8 restricciones = 8 nodos en la red (las ecuaciones de conservación de flujo para las 3 faenas, 2 acopios y los 3 puertos)


## Paso 4. Resolución


In [7]:
res = pyo.SolverFactory("appsi_highs").solve(m)
print("condición de término:", res.solver.termination_condition)
print("valor óptimo: ", pyo.value(m.obj))

flujos = {a: pyo.value(m.x[a]) for a in m.A}
for a, v in sorted(flujos.items()):
    if v > 1e-6:
        print(f"   {a[0]:>10} -> {a[1]:<10} {v:>12,.2f}   de {A[a][2]:,.0f} de capacidad")


condición de término: optimal
valor óptimo:  198000000.0
           A1 -> P1            11,000.00   de 12,000 de capacidad
           A2 -> P2             8,000.00   de 9,000 de capacidad
           A2 -> P3             3,000.00   de 7,000 de capacidad
           F1 -> A1            10,000.00   de 10,000 de capacidad
           F2 -> A1             1,000.00   de 8,000 de capacidad
           F2 -> A2             7,000.00   de 8,000 de capacidad
           F3 -> A2             4,000.00   de 7,000 de capacidad
           F3 -> P3             3,000.00   de 3,000 de capacidad


Observación del óptimo:
El costo mínimo de operación es de $198.000.000 mensuales. \
Este resultado muestra que el modelo funciona con el objetivo de discriminar la ruta en términos económicos. \
F3 despacha la totalidad de su inventario (7.000 t), mientras que F1 y F2 asumen toda la capacidad ociosa de la red (2.000 t y 1.000 t, respectivamente). \
La tonelada marginal extraída en F1 y F2 es la menos competitiva del sistema. \
El solver decide matemáticamente "apagar" ese excedente para proteger la eficiencia global de la operación.

## Paso 5. Verificación de la conservación de flujo

Esta comprobación es obligatoria y vale puntaje. No basta con afirmar que el solver la respetó.


In [8]:
print(f"{'nodo':<10}{'sale':>12}{'entra':>12}{'neto':>12}{'exigido':>12}{'delta q[n]':>12}   estado")
todo_ok = True
for n in N:
    sale  = sum(v for (i, j), v in flujos.items() if i == n)
    entra = sum(v for (i, j), v in flujos.items() if j == n)
    neto  = sale - entra
    delta = q[n] - neto
    ok = abs(neto - q[n]) < 1e-6 or (q[n] > 0 and neto <= q[n] + 1e-6)
    todo_ok &= ok
    print(f"{n:<10}{sale:>12,.2f}{entra:>12,.2f}{neto:>12,.2f}{q[n]:>12,.2f}   {delta:>12,.2f}   {'ok' if ok else 'ERROR'}")
print("\nconservación de flujo verificada en todos los nodos:", todo_ok)


nodo              sale       entra        neto     exigido  delta q[n]   estado
F1           10,000.00        0.00   10,000.00   12,000.00       2,000.00   ok
F2            8,000.00        0.00    8,000.00    9,000.00       1,000.00   ok
F3            7,000.00        0.00    7,000.00    7,000.00           0.00   ok
A1           11,000.00   11,000.00        0.00        0.00           0.00   ok
A2           11,000.00   11,000.00        0.00        0.00           0.00   ok
P1                0.00   11,000.00  -11,000.00  -11,000.00           0.00   ok
P2                0.00    8,000.00   -8,000.00   -8,000.00           0.00   ok
P3                0.00    6,000.00   -6,000.00   -6,000.00           0.00   ok

conservación de flujo verificada en todos los nodos: True


Verificación del balance de masa:
El reporte confirma que el axioma de conservación de flujo se respeta en el 100% de la red. \
Los nodos de transbordo (acopios) presentan un flujo neto nulo, demostrando que operan como paso sin acumular inventario ni generar mermas. \
Los sumideros (puertos) reciben su demanda completa. \
Finalmente, la validación en los orígenes confirma que las 3.000 t/mes de capacidad ociosa fueron retenidas físicamente en F1 (2.000 t) y F2 (1.000 t), validando la decisión geométrica de relajar estas restricciones ($\le$) para mantener el politopo factible.

## Paso 6. Integralidad

¿Salieron enteros los flujos? ¿Por qué? La respuesta debe nombrar la propiedad de la matriz y la
condición sobre el lado derecho, no limitarse a constatar el hecho.


In [ ]:
no_enteros = {a: v for a, v in flujos.items() if abs(v - round(v)) > 1e-6}
print("flujos no enteros:", len(no_enteros))
if no_enteros:
    print(no_enteros)

# TODO: construya la matriz de incidencia nodo-arco y calcule el determinante de al menos
#       tres submatrices cuadradas. Verifique que todos caen en {0, 1, -1}.

import numpy as np

# Construir la matriz de incidencia nodo-arco
nodos_lista = list(N)
arcos_lista = list(A.keys())
matriz = np.zeros((len(nodos_lista), len(arcos_lista)))

for j, (origen, destino) in enumerate(arcos_lista):
    i_orig = nodos_lista.index(origen)
    i_dest = nodos_lista.index(destino)
    matriz[i_orig, j] = 1   # Nodo origen (+)
    matriz[i_dest, j] = -1  # Nodo destino (-)

G = nx.Graph()
G.add_edges_from(arcos_lista)

arbol = nx.minimum_spanning_tree(G)          # cualquier árbol generador sirve, no hace falta que sea "mínimo"
arcos_arbol = set(frozenset(e) for e in arbol.edges())

# índices de columna en 'matriz' que corresponden a arcos del árbol
columnas_arbol = [j for j, (i, k) in enumerate(arcos_lista) if frozenset((i, k)) in arcos_arbol]
print("Arcos del árbol:", [arcos_lista[j] for j in columnas_arbol])
print("N° de arcos del árbol (debe ser N-1 =", len(nodos_lista) - 1, "):", len(columnas_arbol))

# se omite cualquier fila (nodo) — el valor absoluto del determinante no cambia
fila_omitida = 0
filas = [i for i in range(len(nodos_lista)) if i != fila_omitida]


# Extraer cuatro submatrices cuadradas (3x3) para ilustrar la propiedad
submatriz_arbol = matriz[np.ix_(filas, columnas_arbol)]
submatriz_1 = matriz[np.ix_([0, 1, 2], [0, 3, 5])]
submatriz_2 = matriz[np.ix_([0, 1, 2], [0, 6, 7])]
submatriz_3 = matriz[np.ix_([0, 1, 2], [0, 1, 2])]

print("\nVerificación de Unimodularidad Total (Determinantes):")
print("Det submatriz del árbol generador:", round(np.linalg.det(submatriz_arbol)))
print(f"Det Submatriz 1: {round(np.linalg.det(submatriz_1))}")
print(f"Det Submatriz 2: {round(np.linalg.det(submatriz_2))}")
print(f"Det Submatriz 3: {round(np.linalg.det(submatriz_3))}")


flujos no enteros: 0
Arcos del árbol: [('F1', 'A1'), ('F1', 'A2'), ('F2', 'A1'), ('F3', 'A1'), ('A1', 'P2'), ('A1', 'P3'), ('F1', 'P1')]
N° de arcos del árbol (debe ser N-1 = 7 ): 7

Verificación de Unimodularidad Total (Determinantes):
Det submatriz del árbol generador: -1
Det Submatriz 1: 1
Det Submatriz 2: 0
Det Submatriz 3: 0


Análisis de Integralidad:
La ausencia de flujos fraccionarios ratifica la topología del modelo. \
La matriz de restricciones corresponde a la matriz de incidencia de un grafo dirigido, la cual cumple con la propiedad de ***Unimodularidad Total***. \
Los determinantes obtenidos verifican esta condición, ya que toda submatriz cuadrada arroja valores estrictamente en el conjunto $\{-1, 0, 1\}$ (en este caso, matrices singulares con determinante $0$). \
Dado que el vector de flujos exógenos y capacidades está compuesto por valores enteros, el Teorema de Hoffman-Kruskal garantiza matemáticamente que todos los vértices del politopo son enteros, haciendo redundante declarar variables discretas o recurrir a algoritmos de ramificación.   

## Paso 7. Valores duales

Informe cada dual con su **unidad** y declare la **convención de signos** que usa. Un dual sin
unidad no es interpretable y el informe pierde puntaje.


In [10]:
for n in N:
    print(f"{n:<10} {m.dual[m.bal[n]]:>14,.2f}")

# TODO: interprete. ¿Qué significa el dual de un nodo de demanda? ¿Y el de uno con holgura?
# El código cumple con extraer el diccionario de duales almacenado en m.dual


F1                  -0.00
F2                  -0.00
F3                -700.00
A1              -5,100.00
A2              -4,600.00
P1              -7,900.00
P2              -7,900.00
P3             -10,300.00


Interpretación de Valores Duales:
+ Unidad y Convención de Signos: Los duales se miden en pesos por tonelada ($/t). \
La convención de signos del solver indica el impacto en la función objetivo al incrementar el lado derecho ($q_i$) en una unidad. \
Para un nodo de demanda (donde $q_i$ es negativo), exigir una tonelada adicional significa que $q_i$ disminuye en 1 (ej. de -6000 a -6001). \
Por tanto, un dual negativo en un puerto (ej. -10.300) significa que el costo total de la red aumenta en $10.300 por cada tonelada extra que ese puerto exija.
+ Dual en un nodo de demanda: Representa el Costo Marginal Local de abastecimiento. \
Por ejemplo, el dual de P1 refleja que llevar una tonelada marginal hasta ese punto específico de la red, usando la mejor ruta alternativa disponible sin violar las capacidades, le cuesta al sistema exactamente esa cantidad de dinero.
+ Dual en un nodo con holgura (F1 y F2): Su valor es estrictamente cero debido al Teorema de Holgura Complementaria. \
Como las faenas F1 y F2 tienen mineral sobrante (2.000 t y 1.000 t, respectivamente), obligarlas marginalmente a "ofrecer" una tonelada extra no genera ningún valor ni impacto en el costo óptimo de la red, ya que el sistema de transporte ya está rechazando su mineral actual por falta de demanda.

## Paso 8. Análisis de sensibilidad

La forma más segura y más transparente de obtener el valor de una capacidad es **volver a
resolver** con esa capacidad modificada, y comparar. Es lo que hace la función siguiente.


In [20]:
# def resolver(A_mod, q_mod=None):
#     """Resuelve una variante del modelo y devuelve (valor óptimo, flujos)."""
#     q2 = q_mod if q_mod is not None else q
#     mm = pyo.ConcreteModel()
#     mm.A = pyo.Set(initialize=list(A_mod), dimen=2)
#     mm.N = pyo.Set(initialize=list(q2))
#     mm.x = pyo.Var(mm.A, domain=pyo.NonNegativeReals,
#                    bounds=lambda mm, i, j: (A_mod[(i, j)][1], A_mod[(i, j)][2]))
#     mm.obj = pyo.Objective(expr=sum(A_mod[a][0] * mm.x[a] for a in mm.A), sense=pyo.minimize)
#     def bal(mm, n):
#         sale  = sum(mm.x[i, j] for (i, j) in mm.A if i == n)
#         entra = sum(mm.x[i, j] for (i, j) in mm.A if j == n)
#         return sale - entra == q2[n]
#     mm.bal = pyo.Constraint(mm.N, rule=bal)
#     r = pyo.SolverFactory("appsi_highs").solve(mm)
#     if str(r.solver.termination_condition) != "optimal":
#         return None, None
#     return pyo.value(mm.obj), {a: pyo.value(mm.x[a]) for a in mm.A}


# TODO: use resolver() para responder las preguntas de sensibilidad de su instancia.
# Ejemplo de uso: aumentar en una unidad la capacidad de un arco y medir el ahorro.

def resolver(A_mod, q_mod=None):
    q2 = q_mod if q_mod is not None else q
    mm = pyo.ConcreteModel()
    mm.A = pyo.Set(initialize=list(A_mod), dimen=2)
    mm.N = pyo.Set(initialize=list(q2))
    mm.x = pyo.Var(mm.A, domain=pyo.NonNegativeReals,
                   bounds=lambda mm, i, j: (A_mod[(i, j)][1], A_mod[(i, j)][2]))
    mm.obj = pyo.Objective(expr=sum(A_mod[a][0] * mm.x[a] for a in mm.A), sense=pyo.minimize)
    
    def bal(mm, n):
        sale  = sum(mm.x[i, j] for (i, j) in mm.A if i == n)
        entra = sum(mm.x[i, j] for (i, j) in mm.A if j == n)
        # Corrección vital: respetar la capacidad ociosa
        if q2[n] > 0:
            return sale - entra <= q2[n]
        else:
            return sale - entra == q2[n]
            
    mm.bal = pyo.Constraint(mm.N, rule=bal)
    r = pyo.SolverFactory("appsi_highs").solve(mm)
    if str(r.solver.termination_condition) != "optimal":
        return None, None
    return pyo.value(mm.obj), {a: pyo.value(mm.x[a]) for a in mm.A}

costo_base = pyo.value(m.obj)

# 1. Ampliar F1 -> A1 en 1000 t
A_mod_1 = A.copy()
A_mod_1[('F1', 'A1')] = (A[('F1', 'A1')][0], A[('F1', 'A1')][1], A[('F1', 'A1')][2] + 1000)
val_1, flujos_1 = resolver(A_mod_1)
print(f"Ahorro al ampliar F1 -> A1 en 1000 t: ${costo_base - val_1:,.2f}")

# 2. Ampliar F3 -> P3 en 1000 t
A_mod_2 = A.copy()
A_mod_2[('F3', 'P3')] = (A[('F3', 'P3')][0], A[('F3', 'P3')][1], A[('F3', 'P3')][2] + 1000)
val_2, flujos_2 = resolver(A_mod_2)
print(f"Ahorro al ampliar F3 -> P3 en 1000 t: ${costo_base - val_2:,.2f}")



Ahorro al ampliar F1 -> A1 en 1000 t: $900,000.00
Ahorro al ampliar F3 -> P3 en 1000 t: $0.00


### Análisis de Sensibilidad de Arcos Saturados

+ F1 $\rightarrow$ A1 (Ahorro de 900.000): La capacidad extra permite despachar material por una ruta global más eficiente. Al aumentar la capacidad de este arco, el modelo canaliza 1.000 t adicionales por la ruta F1 $\rightarrow$ P1 (costo de $7.000/t) sustituyendo el flujo forzado desde F2 (costo de $7.900/t). La diferencia genera un ahorro marginal de $900 por tonelada.

+ F3 $\rightarrow$ P3 (Ahorro de 0): La capacidad extra en este arco no aporta valor a la función objetivo porque la red posee una ruta alternativa degenerada. Despachar directo cuesta 9.600/t, y despachar a través del acopio (F3 $\rightarrow$ A2 $\rightarrow$ P3) cuesta exactamente lo mismo ($3.900 + $5.700 = $9.600/t). El flujo ya transita por una alternativa idénticamente económica, haciendo redundante la inversión en capacidad.

*NOTA: Se reescribe el codigo de resolver() original, incorporando las líneas de código que definen el criterio de igualdad y desigualdad de los flujos.*

## Paso 9. Respuestas y limitaciones

Responda aquí, en prosa, las preguntas de su instancia. Cierre con la limitación del modelo que se
pide en la sección 4 del enunciado: un supuesto que el modelo hace, que la realidad no cumple, y en
qué dirección sesga la conclusión.

---

**Recordatorio de entrega:** repositorio con `README.md`, `datos/`, `modelo.py`,
`resultados/` y este cuaderno. Domingo 13 de septiembre, 23:59.


### Pregunta 1:
Formule el modelo de flujo a costo mínimo. Justifique por qué las restricciones de las faenas
deben escribirse como desigualdad y no como igualdad.

### 1. Formulación del Modelo y Justificación

**Conjuntos:**
*   $N$: Nodos de la red (Faenas, Acopios, Puertos).
*   $A$: Arcos dirigidos $(i,j)$ que representan las rutas habilitadas.

**Parámetros:**
*   $c_{ij}$: Costo unitario de transporte en el arco $(i,j)$ [$/t].
*   $u_{ij}$: Capacidad máxima del arco $(i,j)$ [t/mes].
*   $q_i$: Flujo exógeno del nodo $i$ [t/mes]. ($q_i > 0$ oferta, $q_i < 0$ demanda, $q_i = 0$ transbordo).

**Variables de decisión:**
*   $x_{ij}$: Cantidad de mineral a transportar por el arco $(i,j)$ [t/mes].

**Función Objetivo:**
Minimizar el costo total mensual de la red logística:
$$ \min Z = \sum_{(i,j) \in A} c_{ij} x_{ij} $$

**Restricciones:**
1. Conservación de flujo (Nodos de Oferta, $q_i > 0$):
$$ \sum_{j:(i,j) \in A} x_{ij} - \sum_{j:(j,i) \in A} x_{ji} \le q_i \quad \forall i \in N_{Oferta} $$

2. Conservación de flujo estricta (Acopios y Puertos, $q_i \le 0$):
$$ \sum_{j:(i,j) \in A} x_{ij} - \sum_{j:(j,i) \in A} x_{ji} = q_i \quad \forall i \in N \setminus N_{Oferta} $$

3. Capacidad y no negatividad:
$$ 0 \le x_{ij} \le u_{ij} \quad \forall (i,j) \in A $$

___
**Justificación de la desigualdad en faenas:**
La red posee un desbalance estructural: la oferta total (28.000 t) supera la demanda total (25.000 t). Si las restricciones de oferta se modelaran como igualdad estricta ($=$), el sistema obligaría a despachar 3.000 t de mineral hacia una red que no tiene capacidad de absorción (sumideros), violando el balance de masa global y volviendo el poliedro matemáticamente infactible (Lema de Farkas). Formularlas como desigualdad ($\le$) permite relajar el sistema, absorbiendo el excedente de 3.000 t como capacidad ociosa directamente en su origen.

In [21]:
# Verificación independiente del costo óptimo (Exigencia 2)
costo_manual = sum(A[a][0] * v for a, v in flujos.items())
print(f"Costo reportado por el solver: ${pyo.value(m.obj):,.2f}")
print(f"Costo calculado manualmente:   ${costo_manual:,.2f}")
print(f"Diferencia:                    ${abs(pyo.value(m.obj) - costo_manual):,.2f}")

Costo reportado por el solver: $198,000,000.00
Costo calculado manualmente:   $198,000,000.00
Diferencia:                    $0.00


___
### Pregunta 2
Resuelva e informe el costo mínimo mensual y el plan de despacho completo

Costo Mínimo y Plan de Despacho

El costo mínimo mensual para la operación de la red logística es de **$198.000.000**. 

El plan de despacho óptimo que permite alcanzar este resultado moviliza exactamente las 25.000 toneladas demandadas, distribuidas de la siguiente manera:

**Desde Faenas (Orígenes):**
*   **F1 $\rightarrow$ A1:** 10.000 t/mes *(satura la capacidad del arco)*
*   **F2 $\rightarrow$ A1:** 1.000 t/mes
*   **F2 $\rightarrow$ A2:** 7.000 t/mes
*   **F3 $\rightarrow$ A2:** 4.000 t/mes
*   **F3 $\rightarrow$ P3:** 3.000 t/mes *(satura la capacidad del arco)*

**Desde Acopios (Transbordo a Destinos):**
*   **A1 $\rightarrow$ P1:** 11.000 t/mes
*   **A2 $\rightarrow$ P2:** 8.000 t/mes
*   **A2 $\rightarrow$ P3:** 3.000 t/mes

___

### Pregunta 3

¿Qué faenas quedan con capacidad ociosa y cuánta en cada caso? Verifique que la suma coincide con el excedente de oferta e interprete el resultado desde la operación

Capacidad Ociosa y Análisis Operacional

El plan de despacho óptimo deja la siguiente capacidad ociosa (mineral no despachado) en los orígenes:
*   **Faena 1 (F1):** 2.000 t/mes (de 12.000 disponibles).
*   **Faena 2 (F2):** 1.000 t/mes (de 9.000 disponibles).
*   **Faena 3 (F3):** 0 t/mes (opera al tope de su capacidad extrayendo 7.000 t).

**Verificación:**
La suma de la capacidad ociosa (2.000 + 1.000 = 3.000 t) coincide de forma exacta con el excedente estructural de la red (28.000 t de oferta agregada frente a 25.000 t de demanda estricta en puertos).

**Interpretación Operacional:**
El modelo optimizador actúa en la práctica como un **discriminador económico**. Que F1 y F2 asuman la retención de inventario revela que la tonelada marginal producida en estas faenas es la menos competitiva del sistema logístico. Dados los costos y los cuellos de botella actuales, movilizar esas 3.000 toneladas hacia los puertos implicaría utilizar las rutas más ineficientes. En consecuencia, la red maximiza su rentabilidad global "apagando" selectivamente la evacuación del mineral con peor posición relativa.
___



### Pregunta 4
Dos arcos quedan saturados en el óptimo. Identifíquelos y determine, para cada uno, cuánto ahorraría la compañía si su capacidad aumentara en 1.000 toneladas mensuales. Los dos resultados no son iguales: explique por qué.

### 4. Sensibilidad de Arcos Saturados y Soluciones Alternativas

Revisando el reporte computacional del solver en la solución óptima, los dos arcos que alcanzan el tope de su capacidad son **F1 $\rightarrow$ A1** (10.000 de 10.000 t) y **F3 $\rightarrow$ P3** (3.000 de 3.000 t). 

(Nota de degeneración: La red presenta soluciones óptimas alternativas idénticas en costo, ya que despachar directo F3 $\rightarrow$ P3 cuesta 9.600/t, lo cual es exactamente igual a despachar F3 $\rightarrow$ A2 $\rightarrow$ P3 por 3.900 + 5.700 = 9.600/t).*

Al evaluar el impacto económico de ampliar en 1.000 t/mes la capacidad mediante la función `resolver()`:

1.  Arco F1 $\rightarrow$ A1 (Ahorro de 900.000/mes):**
    Expandir este canal permite reasignar flujos hacia la vía más competitiva de F1. El sistema despacha 1.000 t adicionales por F1 $\rightarrow$ A1 $\rightarrow$ P1 (4.200 + 2.800 = 7.000/t), liberando y reemplazando 1.000 t que antes se enviaban de forma forzada desde F2 ($5.100 + $2.800 = $7.900/t). El ahorro marginal directo es de $900 por tonelada, sumando un total de 900.000/mes.

2.  Arco F3 $\rightarrow$ P3 (Ahorro de 0/mes):
    Ampliar la capacidad del enlace F3 $\rightarrow$ P3 genera un ahorro de 0 pesos, debido a la presencia de la ruta paralela degenerada vía A2 que ofrece exactamente la misma tarifa global (9.600/t). La red absorbe el flujo por la vía alternativa sin aportar eficiencia o valor marginal adicional al costo total de la operación.

___

### Pregunta 5
Compruebe que todos los flujos óptimos resultan enteros pese a que usted no declaró variables enteras. Explique a qué propiedad de la matriz de restricciones se debe.

### 5. Integralidad y Propiedad de la Matriz

Al evaluar la solución óptima obtenida mediante la relajación lineal (`domain=pyo.NonNegativeReals`), se comprueba que el vector de flujos $x^*$ no contiene componentes fraccionarias; los 13 arcos arrojan valores estrictamente enteros (ej. 10.000, 7.000, 3.000 t).

**Fundamento Matemático:**

Esta propiedad no es una coincidencia numérica, sino una consecuencia estructural del modelo basada en dos pilares:

1.  **Unimodularidad Total (TU):**
    La matriz de restricciones $A$ del problema es la matriz de incidencia nodo-arco de un grafo dirigido. Cada columna de la matriz contiene a lo sumo dos elementos distintos de cero (un $1$ en el nodo de origen y un $-1$ en el nodo de destino). Toda matriz con esta estructura topológica cumple con la propiedad de **Unimodularidad Total**: el determinante de cualquier submatriz cuadrada pertenece estrictamente al conjunto $\{-1, 0, 1\}$.

2.  **Vector de Lado Derecho Entero ($b \in \mathbb{Z}^n$):**
    Los parámetros de oferta exógena en faenas, demandas en puertos y límites superiores de capacidad en los arcos están definidos exclusivamente por números enteros.

Por el **Teorema de Hoffman-Kruskal**, si la matriz de coeficientes $A$ es Totalmente Unimodular y el vector de lados derechos $b$ es entero, la geometría del poliedro factible $P = \{x \mid A x \le b, x \ge 0\}$ garantiza que **todos sus puntos extremos (vértices) tienen coordenadas enteras**. En consecuencia, el algoritmo de optimización lineal siempre encontrará el óptimo en un vértice entero, haciendo completamente innecesario el uso de variables discretas o algoritmos de Ramificación y Acotamiento (*Branch & Bound*).
___

### Pregunta 6
Suponga que el Acopio Calama queda fuera de servicio por mantenimiento mayor. ¿El problema sigue siendo factible? Si no lo es, entregue un argumento que lo demuestre sin recurrir al mensaje del solver

### 6. Análisis de Factibilidad: Indisponibilidad del Acopio Calama (A1)

Si el Acopio Calama (A1) queda completamente fuera de servicio ($x_{A1, j} = 0, x_{i, A1} = 0$), el modelo se vuelve **matemáticamente infactible**.

**Demostración mediante Corte de Red (Teorema de Flujo Máximo / Corte Mínimo):**

Para demostrar la infactibilidad sin apelar a la respuesta del solver, basta realizar un análisis topológico de capacidad sobre el sumidero correspondiente al **Puerto Patillos (P1)**:

1.  **Demanda Estricta de P1:**
    El Puerto 1 exige una recepción obligatoria de $q_{P1} = 11.000\text{ t/mes}$.

2.  **Aislamiento de Entradas:**
    En la topología de la red, los únicos arcos dirigidos que alimentan de forma directa al nodo P1 son:
    *   $\text{A1} \rightarrow \text{P1}$ (Capacidad $u_{A1, P1} = 12.000\text{ t/mes}$)
    *   $\text{F1} \rightarrow \text{P1}$ (Capacidad $u_{F1, P1} = 4.000\text{ t/mes}$)

3.  **Capacidad Máxima Remanente:**
    Al eliminar el Acopio A1, el arco $\text{A1} \rightarrow \text{P1}$ queda inhabilitado (capacidad efectiva $0\text{ t/mes}$). En consecuencia, la capacidad máxima teórica de entrada al nodo P1 se reduce a la única vía directa disponible:
    $$ \text{Capacidad Máxima a P1} = u_{F1, P1} = 4.000\text{ t/mes} $$

4.  **Inviolabilidad del Balance de Masa:**
    Dado que la demanda exigida ($11.000\text{ t/mes}$) supera estrictamente la capacidad física máxima del corte de entrada ($4.000\text{ t/mes}$):
    $$ \sum_{i:(i, P1) \in A} x_{i, P1} \le 4.000 < 11.000 $$

Es físicamente imposible satisfacer el requerimiento de masa de P1. Por tanto, el poliedro de soluciones factibles se reduce a un conjunto vacío ($\emptyset$).
___

### Cierre con la limitación del modelo que se pide en la sección 4 del enunciado: un supuesto que el modelo hace, que la realidad no cumple, y en qué dirección sesga la conclusión.


### 7. Limitaciones Estructurales del Modelo y Sesgo Operacional

**1. Supuesto Restrictivo del Modelo:**
El modelo de flujo a costo mínimo asume **costos marginales constantes y continuidad proporcional** en la función de transporte ($\text{Costo} = c_{ij} \cdot x_{ij}$). Bajo este supuesto, mover 1 tonelada cuesta exactamente la misma proporción que mover 10.000 toneladas, permitiendo el despacho de volúmenes fraccionados o subutilizados en cualquier ruta sin penalización fija.

**2. Divergencia con la Realidad Operativa Minera:**
En la logística minera real (transporte por camiones de alto tonelaje o ferrocarril), los costos operan mediante **tarifas por flete muerto o costos fijos por despacho (setup)**. Contratar una flota o activar una frecuencia ferroviaria implica un costo fijo inicial significativo, independientemente de si el equipo viaja al 30% o al 100% de su capacidad nominal. Los costos unitarios reales disminuyen conforme aumenta el volumen debido a economías de escala.

**3. Dirección del Sesgo en las Conclusiones:**
*   **Subestimación del Costo Real (Sesgo hacia abajo):** Al no incorporar costos fijos por activación de rutas o penalizaciones por bajo llenado, el modelo subestima el costo logístico total que enfrentará la compañía en la práctica.
*   **Atomización Ineficiente de Rutas:** El modelo tiende a recomendar la división de volúmenes pequeños en múltiples rutas alternativas para ajustar capacidades (por ejemplo, enviar volúmenes marginales a través de acopios), cuando en la realidad operacional sería económicamente preferible consolidar todo el tonelaje en una sola ruta principal para maximizar la ocupación de la flota.

# Generación de solucion.csv y duales.csv


In [22]:
import os
import pandas as pd

# Crear el directorio resultados si no existe
os.makedirs("resultados", exist_ok=True)

# 1. Exportar solucion.csv (Flujos óptimos arco por arco)
df_solucion = pd.DataFrame([
    {
        "origen": origen,
        "destino": destino,
        "flujo_t_mes": pyo.value(m.x[origen, destino]),
        "capacidad_max_t": A[(origen, destino)][2],
        "saturado": "SI" if abs(pyo.value(m.x[origen, destino]) - A[(origen, destino)][2]) < 1e-6 else "NO"
    }
    for (origen, destino) in m.A
])
df_solucion.to_csv("resultados/solucion.csv", index=False)
print("Archivo 'resultados/solucion.csv' actualizado con éxito.")

# 2. Exportar duales.csv (Valores duales de los nodos)
df_duales = pd.DataFrame([
    {
        "nodo": n,
        "valor_dual_peso_por_t": m.dual[m.bal[n]],
        "oferta_demanda_exogena": q[n]
    }
    for n in m.N
])
df_duales.to_csv("resultados/duales.csv", index=False)
print("Archivo 'resultados/duales.csv' actualizado con éxito.")

Archivo 'resultados/solucion.csv' actualizado con éxito.
Archivo 'resultados/duales.csv' actualizado con éxito.
